In [1]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer

# ==========================================
# 1. 환경 설정 및 2가지 분석 모드 세팅
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_02"
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P2")
os.makedirs(SAVE_DIR, exist_ok=True)

# 1안(COB 기준)과 2안(3-bit 일괄) 통합 설정
ANALYSIS_MODES = {
    "Mode_1_COB_Baseline": {
        "Llama-3.2-1B-Instruct": {"bf16": "Llama-3.2-1B-Instruct_Original_BF16", "quant": "Llama-3.2-1B-Instruct_GPTQ_8bit", "layers": 16},
        "Qwen2.5-1.5B-Instruct": {"bf16": "Qwen2.5-1.5B-Instruct_Original_BF16", "quant": "Qwen2.5-1.5B-Instruct_GPTQ_3bit", "layers": 28},
        "TinyLlama-1.1B-Chat": {"bf16": "TinyLlama-1.1B-Chat-v1.0_Original_BF16", "quant": "TinyLlama-1.1B-Chat-v1.0_GPTQ_3bit", "layers": 22}
    },
    "Mode_2_Extreme_3Bit": {
        "Llama-3.2-1B-Instruct": {"bf16": "Llama-3.2-1B-Instruct_Original_BF16", "quant": "Llama-3.2-1B-Instruct_GPTQ_3bit", "layers": 16},
        "Qwen2.5-1.5B-Instruct": {"bf16": "Qwen2.5-1.5B-Instruct_Original_BF16", "quant": "Qwen2.5-1.5B-Instruct_GPTQ_3bit", "layers": 28},
        "TinyLlama-1.1B-Chat": {"bf16": "TinyLlama-1.1B-Chat-v1.0_Original_BF16", "quant": "TinyLlama-1.1B-Chat-v1.0_GPTQ_3bit", "layers": 22}
    }
}

TARGET_TOKENS = {"date_idx": [19, 20], "entity_idx": [111, 112, 113, 114]}
    "date_tokens": ["jun", "06", "2026"], 
    "entity_tokens": ["neil", "armstrong", "apollo"]
}

# ==========================================
# 2. 핵심 연산 유틸리티
# ==========================================
def calculate_rogue_and_l2(bf16_tensor, quant_tensor):
    """단일 텐서 쌍에 대한 Global L2 Error 및 Rogue 차원 보정 Cosine 유사도 반환"""
    bf16_mean = bf16_tensor.mean(dim=(0, 1)).float()
    quant_mean = quant_tensor.mean(dim=(0, 1)).float()
    
    l2_error = torch.norm(bf16_mean - quant_mean, p=2).item()
    cos_sim_orig = F.cosine_similarity(bf16_mean.unsqueeze(0), quant_mean.unsqueeze(0)).item()
    
    _, top_indices = torch.topk(torch.abs(bf16_mean), 10)
    mask = torch.ones_like(bf16_mean)
    mask[top_indices] = 0
    
    cos_sim_filtered = F.cosine_similarity((bf16_mean * mask).unsqueeze(0), (quant_mean * mask).unsqueeze(0)).item()
    return l2_error, cos_sim_orig, cos_sim_filtered

# ==========================================
# 3. 메인 분석 엔진
# ==========================================
def run_framework_analysis():
    for mode_name, models in ANALYSIS_MODES.items():
        print(f"\n[{mode_name}] 분석 시작 =========================")
        mode_results = {}
        
        for model_name, config in models.items():
            print(f"  -> {model_name} 처리 중...")
            
            # --- 실 환경 로드 시 아래 주석 해제 ---
            # tokenizer = AutoTokenizer.from_pretrained(model_name)
            # input_ids = torch.load(os.path.join(BASE_DIR, config["bf16"], PROMPT_DIR, "reference_input_ids.pt"))
            # date_idx = find_token_indices(input_ids, tokenizer, TARGET_TOKENS["date_tokens"]) # 이전 코드의 함수 활용
            # entity_idx = find_token_indices(input_ids, tokenizer, TARGET_TOKENS["entity_tokens"])
            
            date_idx, entity_idx = [5, 6], [150, 151, 152] # 테스트용 임시 인덱스
            
            db = {
                "layer": [], 
                "global_l2_attn": [], "global_l2_mlp": [],     # 1단계 (거시) 데이터
                "date_l2_attn": [], "entity_l2_attn": [],      # 2단계 (미시) 데이터
                "date_l2_mlp": [], "entity_l2_mlp": []
            }
            
            for layer in range(config["layers"]):
                # --- 실 환경 텐서 로드 시 아래 주석 해제 ---
                # bf16_attn = torch.load(os.path.join(BASE_DIR, config["bf16"], PROMPT_DIR, "tensors", f"layer_{layer}_attn_output.pt"))
                # quant_attn = torch.load(os.path.join(BASE_DIR, config["quant"], PROMPT_DIR, "tensors", f"layer_{layer}_attn_output.pt"))
                # bf16_mlp = torch.load(os.path.join(BASE_DIR, config["bf16"], PROMPT_DIR, "tensors", f"layer_{layer}_mlp_output.pt"))
                # quant_mlp = torch.load(os.path.join(BASE_DIR, config["quant"], PROMPT_DIR, "tensors", f"layer_{layer}_mlp_output.pt"))
                
                # 테스트용 더미 텐서
                seq_len, dim = 335, 2048
                bf16_attn, quant_attn = torch.randn(1, seq_len, dim), torch.randn(1, seq_len, dim)
                bf16_mlp, quant_mlp = torch.randn(1, seq_len, dim), torch.randn(1, seq_len, dim)

                # [1단계] Global 오차 추출 (시퀀스 전체)
                g_l2_attn, _, _ = calculate_rogue_and_l2(bf16_attn, quant_attn)
                g_l2_mlp, _, _ = calculate_rogue_and_l2(bf16_mlp, quant_mlp)
                
                # [2단계] 특정 토큰 오차 추출 (Slicing)
                d_l2_attn, _, _ = calculate_rogue_and_l2(bf16_attn[:, date_idx, :], quant_attn[:, date_idx, :])
                e_l2_attn, _, _ = calculate_rogue_and_l2(bf16_attn[:, entity_idx, :], quant_attn[:, entity_idx, :])
                d_l2_mlp, _, _ = calculate_rogue_and_l2(bf16_mlp[:, date_idx, :], quant_mlp[:, date_idx, :])
                e_l2_mlp, _, _ = calculate_rogue_and_l2(bf16_mlp[:, entity_idx, :], quant_mlp[:, entity_idx, :])
                
                # 데이터 적재
                db["layer"].append(layer)
                db["global_l2_attn"].append(g_l2_attn)
                db["global_l2_mlp"].append(g_l2_mlp)
                db["date_l2_attn"].append(d_l2_attn)
                db["entity_l2_attn"].append(e_l2_attn)
                db["date_l2_mlp"].append(d_l2_mlp)
                db["entity_l2_mlp"].append(e_l2_mlp)
            
            # CSV 저장
            df = pd.DataFrame(db)
            csv_path = os.path.join(SAVE_DIR, f"{mode_name}_{model_name}.csv")
            df.to_csv(csv_path, index=False)
            mode_results[model_name] = df
            
        # 각 모드 실행 후 시각화 트리거
        plot_step1_macro(mode_name, mode_results)
        plot_step2_micro_llama(mode_name, mode_results["Llama-3.2-1B-Instruct"])

# ==========================================
# 4. 시각화 (다중 패널 고도화)
# ==========================================
def plot_step1_macro(mode_name, results_dict):
    """[1단계] 세 모델 간 Global L2 Error 피크 위치 대조"""
    plt.figure(figsize=(12, 5))
    colors = {"Llama-3.2-1B-Instruct": "blue", "Qwen2.5-1.5B-Instruct": "purple", "TinyLlama-1.1B-Chat": "orange"}
    
    for model_name, df in results_dict.items():
        # Attention의 Global L2 Error를 모델별로 겹쳐 그리기
        plt.plot(df["layer"], df["global_l2_attn"], label=model_name, marker='o', color=colors.get(model_name, "black"))
        
    plt.title(f"[Step 1] Cross-Model Global L2 Error Peak ({mode_name})")
    plt.xlabel("Layer Depth")
    plt.ylabel("Global L2 Error (Attention)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"{mode_name}_Step1_Macro.png"))
    plt.close()

def plot_step2_micro_llama(mode_name, llama_df):
    """[2단계] Llama 3.2 내부의 날짜 토큰 vs 엔티티 토큰 오차 분리 (다중 패널)"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    
    # 패널 1: 언어망 (Attention Block)
    axes[0].plot(llama_df["layer"], llama_df["date_l2_attn"], label='Date Token (Red)', color='red', marker='^')
    axes[0].plot(llama_df["layer"], llama_df["entity_l2_attn"], label='Entity Token (Blue)', color='blue', marker='x')
    axes[0].set_title("Attention Block: Date vs Entity Tokens")
    axes[0].set_xlabel("Layer Depth")
    axes[0].set_ylabel("L2 Error")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.6)
    
    # 패널 2: 논리망 (MLP Block)
    axes[1].plot(llama_df["layer"], llama_df["date_l2_mlp"], label='Date Token (Red)', color='red', marker='^')
    axes[1].plot(llama_df["layer"], llama_df["entity_l2_mlp"], label='Entity Token (Blue)', color='blue', marker='x')
    axes[1].set_title("MLP Block: Date vs Entity Tokens")
    axes[1].set_xlabel("Layer Depth")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.6)
    
    fig.suptitle(f"[Step 2] Llama-3.2-1B Intra-Model Token Disturbance ({mode_name})", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"{mode_name}_Step2_Micro_Llama.png"))
    plt.close()

if __name__ == "__main__":
    run_framework_analysis()
    print(f"\n모든 분석 완료! 결과 데이터와 그래프가 '{SAVE_DIR}'에 저장되었습니다.")


[Mode_1_COB_Baseline] 분석 시작 =========================
  -> Llama-3.2-1B-Instruct 처리 중...
  -> Qwen2.5-1.5B-Instruct 처리 중...
  -> TinyLlama-1.1B-Chat 처리 중...

[Mode_2_Extreme_3Bit] 분석 시작 =========================
  -> Llama-3.2-1B-Instruct 처리 중...
  -> Qwen2.5-1.5B-Instruct 처리 중...
  -> TinyLlama-1.1B-Chat 처리 중...

모든 분석 완료! 결과 데이터와 그래프가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P2'에 저장되었습니다.


In [3]:
import matplotlib as mpl
import os
import shutil

# Matplotlib 폰트 캐시 디렉토리 경로 찾기
cache_dir = mpl.get_cachedir()

# 캐시 디렉토리 삭제 (초기화)
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Matplotlib 폰트 캐시가 성공적으로 삭제되었습니다. 커널을 재시작해 주세요.")
else:
    print("삭제할 폰트 캐시가 없습니다.")

Matplotlib 폰트 캐시가 성공적으로 삭제되었습니다. 커널을 재시작해 주세요.


In [1]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 시각화 스타일 설정 (Seaborn rc 파라미터 통합 세팅)
sns.set_theme(
    style="whitegrid", 
    palette="muted", 
    rc={
        "font.family": "Malgun Gothic",  # Mac 사용 시 "AppleGothic"으로 변경
        "axes.unicode_minus": False      # 유니코드 마이너스 기호 사용 안 함 (경고 해결 핵심)
    }
)

# ==========================================
# 1. 환경 설정 및 분석 타겟 레이어 세팅
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_02"
SAVE_DIR = os.path.join(BASE_DIR, "Activation_Analysis_Results")
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_LAYER = 8 
TARGET_TOKENS = {"date_idx": [5, 6], "entity_idx": [150, 151, 152]}

# ==========================================
# 2. 순수 실데이터 로드 로직 (방안 A)
# ==========================================
def load_real_tensors():
    """실제 양자화 및 BF16 텐서를 로드하고 배치 차원을 제거합니다."""
    
    path_bf16 = os.path.join(BASE_DIR, "Llama-3.2-1B-Instruct_Original_BF16", PROMPT_DIR, "tensors", f"layer_{TARGET_LAYER}_attn_output.pt")
    path_3bit = os.path.join(BASE_DIR, "Llama-3.2-1B-Instruct_GPTQ_3bit", PROMPT_DIR, "tensors", f"layer_{TARGET_LAYER}_attn_output.pt")
    path_2bit = os.path.join(BASE_DIR, "Llama-3.2-1B-Instruct_GPTQ_2bit", PROMPT_DIR, "tensors", f"layer_{TARGET_LAYER}_attn_output.pt")
    
    # [batch, seq_len, dim] -> [0] 인덱싱을 통해 [seq_len, dim] 2D 텐서로 변환 (시각화를 위한 필수 조치)
    bf16 = torch.load(path_bf16)[0]
    quant_3bit = torch.load(path_3bit)[0]
    quant_2bit = torch.load(path_2bit)[0]
    
    return bf16, quant_3bit, quant_2bit

# ==========================================
# 3. 다층적 시각화 함수
# ==========================================
def plot_1_l0_l1_reversal(bf16, quant_3bit, quant_2bit):
    plt.figure(figsize=(10, 6))
    sns.kdeplot(bf16.flatten().numpy(), label="BF16", color="black", linestyle="--")
    sns.kdeplot(quant_3bit.flatten().numpy(), label="3-bit", color="blue")
    sns.kdeplot(quant_2bit.flatten().numpy(), label="2-bit", color="red", alpha=0.5)
    plt.xlim(-10, 10)
    plt.title(f"Layer {TARGET_LAYER}: Activation Distribution (L0 vs L1)", fontsize=14)
    plt.xlabel("Activation Value")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig1_Real_Reversal_KDE.png"), dpi=300)
    plt.close()

def plot_2_rogue_dimension_scatter(bf16, quant_3bit):
    date_bf16 = bf16[TARGET_TOKENS["date_idx"], :].mean(dim=0).abs().numpy()
    date_3bit = quant_3bit[TARGET_TOKENS["date_idx"], :].mean(dim=0).abs().numpy()
    
    plt.figure(figsize=(12, 5))
    plt.scatter(range(2048), date_bf16, alpha=0.5, label="BF16", color="gray", s=10)
    plt.scatter(range(2048), date_3bit, alpha=0.7, label="3-bit", color="red", s=15)
    plt.yscale('symlog') 
    plt.title(f"Layer {TARGET_LAYER}: Dimension-wise Absolute Activation (Date Tokens)", fontsize=14)
    plt.xlabel("Hidden Dimension Index")
    plt.ylabel("Absolute Activation Value (Log Scale)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig2_Real_Rogue_Scatter.png"), dpi=300)
    plt.close()

def plot_3_cumulative_energy(quant_3bit):
    date_3bit = quant_3bit[TARGET_TOKENS["date_idx"], :].mean(dim=0).abs()
    sorted_vals, _ = torch.sort(date_3bit, descending=True)
    cumulative_energy = torch.cumsum(sorted_vals, dim=0) / torch.sum(sorted_vals)
    
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, 2049), cumulative_energy.numpy(), color='purple', linewidth=2)
    plt.title("Cumulative Energy Distribution (3-bit Date Tokens)", fontsize=14)
    plt.xlabel("Number of Dimensions (Sorted)")
    plt.ylabel("Cumulative Energy Ratio")
    plt.xlim(0, 100)
    plt.grid(True, which="both", ls="--")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig3_Real_Energy_CDF.png"), dpi=300)
    plt.close()

def plot_4_token_energy_violin(quant_3bit):
    date_vals = quant_3bit[TARGET_TOKENS["date_idx"], :].flatten().numpy()
    entity_vals = quant_3bit[TARGET_TOKENS["entity_idx"], :].flatten().numpy()
    
    df = pd.DataFrame({
        "Activation": np.concatenate([date_vals, entity_vals]),
        "Token Type": ["Date Tokens"] * len(date_vals) + ["Entity Tokens"] * len(entity_vals)
    })
    
    plt.figure(figsize=(8, 6))
    sns.violinplot(data=df, x="Token Type", y="Activation", hue="Token Type", palette=["red", "blue"], inner="quartile", legend=False)
    plt.title("Activation Spread: Date vs Entity Tokens (3-bit)", fontsize=14)
    plt.ylabel("Activation Value")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4_Real_Token_Violin.png"), dpi=300)
    plt.close()

def plot_5_activation_heatmap(quant_3bit):
    date_mean = quant_3bit[TARGET_TOKENS["date_idx"], :].mean(dim=0)
    _, top_indices = torch.topk(date_mean.abs(), 50)
    
    heatmap_data = torch.stack([
        quant_3bit[TARGET_TOKENS["date_idx"][0], top_indices],
        quant_3bit[TARGET_TOKENS["date_idx"][1], top_indices],
        quant_3bit[TARGET_TOKENS["entity_idx"][0], top_indices],
        quant_3bit[TARGET_TOKENS["entity_idx"][1], top_indices],
    ]).numpy()
    
    plt.figure(figsize=(12, 4))
    sns.heatmap(heatmap_data, cmap="coolwarm", center=0, 
                yticklabels=["Date_1", "Date_2", "Entity_1", "Entity_2"],
                cbar_kws={'label': 'Activation Magnitude'})
    plt.title("Top 50 Activation Dimensions Heatmap", fontsize=14)
    plt.xlabel("Top 50 Hidden Dimensions")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig5_Real_Token_Heatmap.png"), dpi=300)
    plt.close()

# ==========================================
# 4. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print(f"Layer {TARGET_LAYER} 실제 텐서 데이터 로드 중...")
    bf16, quant_3bit, quant_2bit = load_real_tensors()
    
    print("다층적 시각화 생성 중...")
    plot_1_l0_l1_reversal(bf16, quant_3bit, quant_2bit)
    plot_2_rogue_dimension_scatter(bf16, quant_3bit)
    plot_3_cumulative_energy(quant_3bit)
    plot_4_token_energy_violin(quant_3bit)
    plot_5_activation_heatmap(quant_3bit)
    
    print(f"완료! 5개의 시각화 이미지가 '{SAVE_DIR}'에 정상 저장되었습니다.")

Matplotlib is building the font cache; this may take a moment.


Layer 8 실제 텐서 데이터 로드 중...
다층적 시각화 생성 중...


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], 

완료! 5개의 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Activation_Analysis_Results'에 정상 저장되었습니다.
